In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

from xgboost import XGBRegressor
from xgboost import plot_importance

from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

import joblib
import requests
from datetime import datetime, timedelta

In [97]:
df = pd.read_csv('data/delhi-weather-aqi-2025.csv')
df.head()

,date_ist,time_ist,location,lat,lon,temp_c,humidity,pressure_mb,windspeed_kph,condition_text,description,aqi_index,pm2_5,pm10,co,no2
0,01/01/2025,0:00,Anand Vihar,28.6469,77.316,8.1,100,995.4,2.9,Mainly clear,WMO Code 1,197,185.8,188.6,1907,56.7
1,01/01/2025,1:00,Anand Vihar,28.6469,77.316,7.7,100,994.7,3.2,Overcast,WMO Code 3,198,174.6,177.4,1669,44.8
2,01/01/2025,2:00,Anand Vihar,28.6469,77.316,7.5,100,994.3,4.5,Overcast,WMO Code 3,199,164.4,166.7,1493,34.6
3,01/01/2025,3:00,Anand Vihar,28.6469,77.316,7.8,99,994.1,6.0,Overcast,WMO Code 3,200,156.5,158.8,1401,26.7
4,01/01/2025,4:00,Anand Vihar,28.6469,77.316,7.3,100,993.8,6.8,Overcast,WMO Code 3,200,149.5,151.8,1372,20.6


In [98]:
df['datetime'] = pd.to_datetime(df['date_ist'] + ' ' + df['time_ist'], format= '%d/%m/%Y %H:%M')
df = df.set_index(df['datetime']).sort_index()
df.head()

,date_ist,time_ist,location,lat,lon,temp_c,humidity,pressure_mb,windspeed_kph,condition_text,description,aqi_index,pm2_5,pm10,co,no2,datetime
datetime,,,,,,,,,,,,,,,,,
2025-01-01,01/01/2025,0:00,Anand Vihar,28.6469,77.3160,8.1,100,995.4,2.9,Mainly clear,WMO Code 1,197,185.8,188.6,1907,56.7,2025-01-01
2025-01-01,01/01/2025,0:00,Okhla Phase III,28.5273,77.2618,8.6,100,992.4,4.1,Clear sky,WMO Code 0,197,193.7,197.0,1645,74.2,2025-01-01
2025-01-01,01/01/2025,0:00,Dwarka,28.5882,77.0494,8.0,100,994.4,2.8,Overcast,WMO Code 3,197,193.7,197.0,1724,74.2,2025-01-01
2025-01-01,01/01/2025,0:00,IGI Airport,28.5562,77.1000,8.0,100,991.6,2.8,Overcast,WMO Code 3,197,193.7,197.0,1688,74.2,2025-01-01
2025-01-01,01/01/2025,0:00,Connaught Place,28.6304,77.2177,8.2,100,993.8,1.8,Overcast,WMO Code 3,197,185.8,188.6,1645,56.7,2025-01-01


In [99]:
# Pick one location and model it separately
df = df[df['location'] == 'Anand Vihar'].copy()

### Fetching latest data too

In [ ]:
def get_openmeteo_aqi(start_date, end_date):
    url = (
        "https://air-quality-api.open-meteo.com/v1/air-quality"
        f"?latitude=28.6469"
        f"&longitude=77.3168"
        f"&hourly=pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,us_aqi"
        f"&start_date={start_date}"
        f"&end_date={end_date}"
        f"&timezone=Asia/Kolkata"
    )

    r = requests.get(url).json()
    hourly = r['hourly']

    # Build DataFrame directly from response
    df_new = pd.DataFrame({
        'datetime':   pd.to_datetime(hourly['time']),
        'aqi_index':  hourly['us_aqi'],
        'pm2_5':      hourly['pm2_5'],
        'pm10':       hourly['pm10'],
        'co':         hourly['carbon_monoxide'],
        'no2':        hourly['nitrogen_dioxide'],
    })

    df_new = df_new.set_index('datetime')
    return df_new


def get_openmeteo_weather(start_date, end_date):
    url = (
        "https://archive-api.open-meteo.com/v1/archive"
        f"?latitude=28.6469"
        f"&longitude=77.3168"
        f"&hourly=temperature_2m,relative_humidity_2m,pressure_msl,wind_speed_10m"
        f"&start_date={start_date}"
        f"&end_date={end_date}"
        f"&timezone=Asia/Kolkata"
        f"&wind_speed_unit=kmh"
    )

    r = requests.get(url).json()
    hourly = r['hourly']

    df_weather = pd.DataFrame({
        'datetime':      pd.to_datetime(hourly['time']),
        'temp_c':        hourly['temperature_2m'],
        'humidity':      hourly['relative_humidity_2m'],
        'pressure_mb':   hourly['pressure_msl'],
        'windspeed_kph': hourly['wind_speed_10m'],
    })

    df_weather = df_weather.set_index('datetime')
    return df_weather


# Fetch both and merge
aqi_data     = get_openmeteo_aqi('2026-01-01', '2026-04-05')
weather_data = get_openmeteo_weather('2026-01-01', '2026-04-05')

# Merge on datetime index
new_data = pd.concat([aqi_data, weather_data], axis=1)
df = pd.concat([df, new_data])

new_data.head()

                     aqi_index  pm2_5   pm10      co   no2  temp_c  humidity  \
datetime                                                                       
2026-01-01 00:00:00        219  149.5  149.9  1071.0  33.1    11.1        97   
2026-01-01 01:00:00        217  144.4  145.3  1024.0  26.3    10.8        98   
2026-01-01 02:00:00        215  140.6  141.1   997.0  21.2    10.4        98   
2026-01-01 03:00:00        213  137.4  137.8   995.0  17.4    10.2        99   
2026-01-01 04:00:00        210  134.6  135.5  1013.0  15.3    10.1        99   

                     pressure_mb  windspeed_kph  
datetime                                         
2026-01-01 00:00:00       1013.6            6.9  
2026-01-01 01:00:00       1013.7            7.6  
2026-01-01 02:00:00       1013.8            6.2  
2026-01-01 03:00:00       1013.9            6.0  
2026-01-01 04:00:00       1013.7            5.1  


In [116]:
fig = px.line(
    df,
    x=df.index,
    y= 'aqi_index',
    markers=True
)
fig.show()

## Feature Engineering

In [ ]:
df['hour'] = df.index.hour
df['day_of_week'] = df.index.day_of_week
df['month'] = df.index.month
df['is_weekend'] = (df.index.day_of_week > 5).astype(int)

df['season'] = df['month'].map({
    12:0, 1:0, 2:0,    # winter
     3:1, 4:1, 5:1,    # spring
     6:2, 7:2, 8:2,    # summer
     9:3,10:3,11:3     # autumn
})

df['is_rush_hour'] = df['hour'].isin([7,8,9,17,18,19]).astype(int)

In [103]:
# AQI Lags

df['aqi_lag_1h'] = df['aqi_index'].shift(1)
df['aqi_lag_24h'] = df['aqi_index'].shift(24)
df['aqi_lag_168h'] = df['aqi_index'].shift(168) # Last week same hour

# pollutants lags

df['pm2_5_lag_24h'] = df['pm2_5'].shift(24)
df['pm10_lag_24h'] = df['pm10'].shift(24)
df['co_lag_24h'] = df['co'].shift(24)
df['no2_lag_24h'] = df['no2'].shift(24)

# Other LAgs

df['wind_lag_1h'] = df['windspeed_kph'].shift(1)
df['humidity_lag_1h'] = df['humidity'].shift(1)
df['temp_lag_1h'] = df['temp_c'].shift(1)

# High wind disperses pollution
df['wind_aqi'] =  df['wind_lag_1h'] * df['aqi_lag_1h']

# MA and std
df['aqi_MA_24h'] = df['aqi_index'].rolling(window=24).mean().shift(1)
df['aqi_MA_168h'] = df['aqi_index'].rolling(window=168).mean().shift(1)

df['aqi_std_24h'] = df['aqi_index'].rolling(window=24).std().shift(1) # Volatility

In [ ]:
df['target_aqi'] = df['aqi_index'].shift(-1) # Target Column

In [ ]:
# These add no predictive value, hence deleting them
df = df.drop(columns=[
    'date_ist', 'time_ist', 'location', 
    'lat', 'lon', 'description', 'condition_text',
    'datetime'
])

In [ ]:
df = df.dropna() # dropping the NaN's

### Train/Test split

In [ ]:
predictors = [
    'hour', 'day_of_week', 'month',
    'is_weekend', 'season', 'is_rush_hour', 'aqi_lag_1h', 'aqi_lag_24h',
    'aqi_lag_168h', 'pm2_5_lag_24h', 'pm10_lag_24h', 'co_lag_24h',
    'no2_lag_24h', 'wind_lag_1h', 'humidity_lag_1h', 'temp_lag_1h',
    'wind_aqi', 'aqi_MA_24h', 'aqi_MA_168h', 'aqi_std_24h'
]

target = 'target_aqi'

split = int(len(df.index)*0.8) # 80:20 Ratio in time order

train = df[:split]
test = df[split:]

X_train = train[predictors]
X_test = test[predictors]

y_train = train[target]
y_test = test[target]

In [110]:
def forecast(model, X_train, X_test, y_train, y_test, model_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print('='*40)
    print(f'{model_name} Model')
    print('='*40)

    print(f'MAE: {mean_absolute_error(y_test, y_pred):.3f}')
    print(f'Percentage Error: {mean_absolute_percentage_error(y_test, y_pred)*100:.2f}')
    print('\n')

    return y_pred,model

In [ ]:
# Extracting y_pred and model fro further evaluation
y_pred, model= forecast(XGBRegressor(n_estimators = 1000, learning_rate = 0.1, gamma= 0.1, max_depth=6, n_jobs=-1, random_state=42), X_train, X_test, y_train, y_test, 'XGBoost Regressor')

XGBoost Regressor Model
MAE: 4.244
Percentage Error: 1.92




In [ ]:
results = {
    "Model": ["XGBoost", "Linear Regression", "KNN", "Random Forest"],
    "MAE": [4.244, 5.981, 28.509, 3.584],
    "Percentage Error": [1.92, 3.58, 11.45, 1.74]
}

# Convert to DataFrame
df_results = pd.DataFrame(results)

# Plot MAE comparison
fig_mae = px.bar(
    df_results,
    x="Model",
    y="MAE",
    color="Model",
    text="MAE",
    title="Model Comparison: Mean Absolute Error"
)
fig_mae.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig_mae.show()

# Plot Percentage Error comparison
fig_pct = px.bar(
    df_results,
    x="Model",
    y="Percentage Error",
    color="Model",
    text="Percentage Error",
    title="Model Comparison: Percentage Error"
)
fig_pct.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig_pct.show()

### Plotting Feature Imporatnces

In [ ]:
# Extracting importance
importance = model.get_booster().get_score(importance_type='weight')

# Converting to DataFrame
df_importance = pd.DataFrame({
    'Feature': list(importance.keys()),
    'Importance': list(importance.values())
}).sort_values(by='Importance', ascending=True)

# Plotting
fig = px.bar(
    df_importance,
    x='Importance',
    y='Feature',
    orientation='h',
    title="XGBoost Feature Importance",
    text='Importance'
)
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.show()


### Actual vs Predicted values

In [ ]:
# Building DataFrame
df_results = pd.DataFrame({
    "Actual AQI": y_test,
    "Predicted AQI": y_pred
})

# Using index as x-axis
df_results["Index"] = range(len(df_results))

fig = px.scatter(
    df_results,
    x="Index",
    y=["Actual AQI", "Predicted AQI"],  # plotting both columns
    title="Actual vs Predicted AQI"
)

fig.show()

# Saving Model

In [114]:
# Save trained model
joblib.dump(model, 'xgboost_aqi_model.pkl')

# Save your history data for lag features
df.to_csv('data/AQI(With Features).csv', index=True)